# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1. 2. 3.] [0 1 1] Деление / 2 делит каждый элемент массива на 2, а // 4 выполняет целочисленное деление каждого элемента на 4
2. [4. 4. 4.] np.ones((3, 4)) создаёт матрицу из трёх строк и четырёх столбцов, заполненную единицами. sum(axis=1) складывает элементы каждой строки
3. [[0 1 2]
 [1 2 3]
 [2 3 4]]
 NumPy складывает два массива поэлементно, используя broadcasting. Первый массив имеет форму (3,), второй (3, 1), поэтому получается матрица 3 на 3
4. 6 p.sqrt(x) извлекает квадратный корень из каждого элемента: [1., 2., 3.]. Метод sum() складывает их и получает 6
5. 7.5 np.arange(1, 5) создаёт массив [1, 2, 3, 4]. Возведение в квадрат даёт [1, 4, 9, 16]. Среднее значение рассчитывается как (1 + 4 + 9 + 16) / 4 = 7.5

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [4]:
# 1
sep  = amount[clean & (month == "2026-09")].sum()
octo = amount[clean & (month == "2026-10")].sum()
print(clean.sum(), sep, octo, (octo / sep - 1) * 100)

4739 35243410.0 34419520.0 -2.337713632137184


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [5]:
# 2
cats = np.unique(cat)
months = np.unique(month)
rev = np.zeros((len(cats), 2))
for i, c in enumerate(cats):
    for j, m in enumerate(months):
        rev[i, j] = amount[clean & (cat == c) & (month == m)].sum()
print(rev)
total = rev.sum(axis=1)
print(cats[total.argmax()])
chg = (rev[:, 1] / rev[:, 0] - 1) * 100
print(chg.round(1), cats[chg.argmin()])

[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Электроника
[ 2.6  7.1  2.6 -7.2 -2.5] Одежда


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [6]:
# 3
cnt = np.zeros((len(cats), 2))
for i, c in enumerate(cats):
    for j, m in enumerate(months):
        cnt[i, j] = (clean & (cat == c) & (month == m)).sum()
avg = rev / cnt
print(avg.round(0))
print(cats[avg[:, 1].argmax()])

[[10499. 10768.]
 [ 4020.  3816.]
 [ 6187.  6159.]
 [13521. 13046.]
 [32773. 33600.]]
Электроника


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [7]:
# 4
cities = np.unique(city)
table = np.zeros((len(cities), 2))
for i, c in enumerate(cities):
    for j, m in enumerate(months):
        table[i, j] = amount[clean & (city == c) & (month == m)].sum()
share = table / table.sum(axis=0)
print(share.round(4), share.sum(axis=0))

[[0.0731 0.0738]
 [0.4115 0.4254]
 [0.2563 0.2571]
 [0.0262 0.0282]
 [0.1035 0.098 ]
 [0.1293 0.1175]] [1. 1.]


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [8]:
# 5
d = table[:, 1] - table[:, 0]
contrib = d / d.sum() * 100
print(dict(zip(cities, contrib.round(1))))

{np.str_('Актобе'): np.float64(4.4), np.str_('Алматы'): np.float64(-17.2), np.str_('Астана'): np.float64(22.3), np.str_('Атырау'): np.float64(-5.6), np.str_('Караганда'): np.float64(33.6), np.str_('Шымкент'): np.float64(62.5)}


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [9]:
# 6
share = table / table.sum(axis=1, keepdims=True)
print(share.round(3), share.sum(axis=1))

[[0.504 0.496]
 [0.498 0.502]
 [0.505 0.495]
 [0.488 0.512]
 [0.52  0.48 ]
 [0.53  0.47 ]] [1. 1. 1. 1. 1. 1.]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [10]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()
n = (np.abs(z) > 3).sum()
print(n, n / len(x) * 100, z.mean(), z.std())

87 1.8358303439544206 7.04695264417065e-17 1.0


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [11]:
# 8
m8 = clean & ~np.isnan(days)
X = np.column_stack([amount[m8], days[m8]])
Xs = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print(X.shape, Xs.min(axis=0), Xs.max(axis=0))

(4595, 2) [0. 0.] [1. 1.]
